# 01 · Data Audit

**Tujuan audit**: memastikan data *layak dipercaya* sebelum EDA dan feature engineering. Notebook ini **tidak** mencari pola prediktif (itu tugas EDA), tapi menjawab:

| # | Pertanyaan | Kenapa penting |
|---|---|---|
| 1 | Ukuran & rentang waktu tiap tabel wajar? | Gambaran awal skala data |
| 2 | Null hanya ada di kolom yang memang boleh null? | Null tak terduga = data rusak |
| 3 | Kunci unik & tidak ada duplikat? | Duplikat menggelembungkan agregasi |
| 4 | Nilai kategori & rentang angka sesuai kamus data? | Nilai aneh = bug / outlier |
| 5 | Relasi antar tabel utuh? | Join yang bocor/hilang merusak fitur |
| 6 | Aturan bisnis internal konsisten? | Memahami **semantik** kolom (mis. arti DPD) |
| 7 | Tidak ada informasi setelah `decision_datetime`? | **Kebocoran data** |
| 8 | Cakupan tiap tabel sama di train & test? | Pola missing harus konsisten |
| 9 | Distribusi train vs test mirip? | Validasi silang bisa dipercaya |
| 10 | Seberapa ekstrem outlier-nya? | Persiapan transformasi fitur |

Setiap pengecekan dicatat ke tabel `CHECKS` dengan status ✅ PASS / ⚠️ WARN / ❌ FAIL, lalu dirangkum di bagian akhir dan disimpan ke `reports/audit_results.csv`.

> Prasyarat: jalankan `00_csv_to_parquet.ipynb` terlebih dahulu.

## 0. Setup

In [1]:
from pathlib import Path
import json
import numpy as np
import polars as pl
from scipy.stats import ks_2samp

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PQ_DIR = ROOT / "data" / "parquet"
REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_hide_dataframe_shape(True)

manifest = json.loads((PQ_DIR / "_manifest.json").read_text())
print("polars", pl.__version__, "| e-commerce sumber:", manifest["ecommerce_source"],
      "(SAMPLING)" if manifest["ecommerce_is_sampled"] else "(PENUH)")

rd = lambda name: pl.read_parquet(PQ_DIR / f"{name}.parquet")
tr = rd("train_applications")
te = rd("test_applications")
ss = rd("sample_submission")
cp = rd("customer_profiles")
pa = rd("previous_applications")
rh = rd("repayment_history")
ab = rd("account_behavior")
bo = rd("bureau_observations")
# e-commerce tabel terbesar → dibaca lazy; hanya hasil agregasi yang masuk RAM
ec = pl.scan_parquet(PQ_DIR / "ecommerce_transactions.parquet")

# Gabungan aplikasi terpilih (train+test) + cutoff per nasabah
app = pl.concat([
    tr.drop("target").with_columns(split=pl.lit("train")),
    te.with_columns(split=pl.lit("test")),
])
CUT = app.select("customer_id", "split", pl.col("decision_datetime").alias("cutoff"))
print(f"train {tr.height:,} | test {te.height:,} | total nasabah terpilih {app.height:,}")

polars 2.0.0 | e-commerce sumber: ecommerce_transactions.csv (PENUH)
train 70,000 | test 30,000 | total nasabah terpilih 100,000


In [2]:
CHECKS = []
ICON = {"PASS": "✅", "WARN": "⚠️ ", "FAIL": "❌"}

def check(section, table, desc, n_bad, n_total=None, severity="FAIL", note=""):
    """Catat satu pengecekan. n_bad = jumlah baris yang melanggar aturan."""
    n_bad = int(n_bad)
    status = "PASS" if n_bad == 0 else severity
    pct = n_bad / n_total * 100 if n_total else None
    CHECKS.append(dict(section=section, table=table, check=desc, status=status, n_bad=n_bad,
                       n_total=n_total, pct_bad=None if pct is None else round(pct, 4), note=note))
    extra = ""
    if n_bad:
        extra = f"  ({n_bad:,}/{n_total:,} = {pct:.3f}%)" if n_total else f"  ({n_bad:,})"
        if note:
            extra += f" — {note}"
    print(f"{ICON[status]} [{table}] {desc}{extra}")

def nbad(df, expr):
    """Jumlah baris di mana expr bernilai True (df eager atau lazy)."""
    out = df.select(expr.fill_null(False).sum())
    return (out.collect() if isinstance(out, pl.LazyFrame) else out).item()

def nrows(df):
    return df.select(pl.len()).collect().item() if isinstance(df, pl.LazyFrame) else df.height

def n_dup(df, cols):
    """Jumlah baris berlebih untuk kombinasi kolom (0 = unik)."""
    uniq = df.select(cols).unique().select(pl.len())
    uniq = uniq.collect().item() if isinstance(uniq, pl.LazyFrame) else uniq.item()
    return nrows(df) - uniq

def days_between(later, earlier):
    """Selisih hari dibulatkan ke bawah (aman untuk polars 1.x & 2.x)."""
    return ((later - earlier).dt.total_seconds() // 86400).cast(pl.Int64)

## 1. Gambaran umum: ukuran & rentang waktu

In [3]:
TIME_COL = {
    "train_applications": (tr, "decision_datetime"),
    "test_applications": (te, "decision_datetime"),
    "customer_profiles": (cp, None),
    "previous_applications": (pa, "decision_datetime"),
    "repayment_history": (rh, "due_date"),
    "account_behavior": (ab, "month"),
    "bureau_observations": (bo, "report_datetime"),
    "ecommerce_transactions": (ec, "transaction_datetime"),
}
rows = []
for name, (df, tcol) in TIME_COL.items():
    aggs = [pl.len().alias("rows"), pl.col("customer_id").n_unique().alias("customers")]
    if tcol:
        aggs += [pl.col(tcol).min().cast(pl.Datetime("us")).dt.strftime("%Y-%m-%d %H:%M").alias("min_time"),
                 pl.col(tcol).max().cast(pl.Datetime("us")).dt.strftime("%Y-%m-%d %H:%M").alias("max_time")]
    r = df.select(aggs)
    r = (r.collect() if isinstance(r, pl.LazyFrame) else r).to_dicts()[0]
    width = len(df.collect_schema()) if isinstance(df, pl.LazyFrame) else df.width
    rows.append({"table": name, "cols": width, **r,
                 "rows_per_customer": round(r["rows"] / r["customers"], 2)})
overview = pl.DataFrame(rows)
overview

table,cols,rows,customers,min_time,max_time,rows_per_customer
str,i64,i64,i64,str,str,f64
"""train_applications""",10,70000,70000,"""2025-01-01 08:00""","""2025-12-28 20:59""",1.0
"""test_applications""",9,30000,30000,"""2025-01-01 08:07""","""2025-12-28 20:52""",1.0
"""customer_profiles""",8,100000,100000,null,null,1.0
"""previous_applications""",11,201984,70000,"""2023-01-04 13:33""","""2025-12-12 17:10""",2.89
"""repayment_history""",9,1361341,66157,"""2023-02-04 13:33""","""2025-12-28 15:08""",20.58
"""account_behavior""",10,1084418,70000,"""2023-02-01 00:00""","""2025-11-01 00:00""",15.49
"""bureau_observations""",14,797563,100000,"""2022-09-11 10:32""","""2025-12-28 18:38""",7.98
"""ecommerce_transactions""",10,4359817,100000,"""2023-01-02 13:15""","""2025-12-27 19:59""",43.6


In [4]:
# Target: hanya 0/1, berapa proporsinya, stabil per bulan?
check("1-overview", "train", "target hanya bernilai 0/1", nbad(tr, ~pl.col("target").is_in([0, 1])), tr.height)
print(f"\nTarget rate: {tr['target'].mean():.4%}  ({tr['target'].sum():,} positif dari {tr.height:,})")
(tr.group_by(pl.col("decision_datetime").dt.strftime("%Y-%m").alias("bulan_keputusan"))
   .agg(pl.len().alias("n"), pl.col("target").mean().round(4).alias("target_rate"))
   .sort("bulan_keputusan")
   .join(te.group_by(pl.col("decision_datetime").dt.strftime("%Y-%m").alias("bulan_keputusan"))
           .agg(pl.len().alias("n_test")), on="bulan_keputusan", how="left"))

✅ [train] target hanya bernilai 0/1

Target rate: 13.5814%  (9,507 positif dari 70,000)


bulan_keputusan,n,target_rate,n_test
str,u32,f64,u32
"""2025-01""",5778,0.143,2610
"""2025-02""",5898,0.1473,2477
"""2025-03""",5809,0.1384,2580
"""2025-04""",5910,0.1308,2459
"""2025-05""",5864,0.1201,2439
"""2025-06""",5776,0.1354,2513
"""2025-07""",5739,0.1289,2478
"""2025-08""",5809,0.1362,2496
"""2025-09""",5830,0.1266,2485


## 2. Null & string kosong

Kamus data menyebut kolom mana yang **boleh** null. Null di luar daftar itu = masalah.

In [5]:
NULLABLE = {
    "previous_applications": {"loan_id", "approved_amount", "annual_interest_rate", "scheduled_monthly_installment"},
    "repayment_history": {"last_payment_datetime"},
    "bureau_observations": {"score", "utilization"},
}
null_rows = []
for name, (df, _) in TIME_COL.items():
    lf = df.lazy() if isinstance(df, pl.DataFrame) else df
    total = nrows(lf)
    nulls = lf.select(pl.all().null_count()).collect().to_dicts()[0]
    allowed = NULLABLE.get(name, set())
    bad = sum(v for c, v in nulls.items() if c not in allowed)
    check("2-null", name, "tidak ada null di kolom non-nullable", bad, total)
    for c in allowed:
        null_rows.append({"table": name, "column": c, "null_count": nulls[c],
                          "null_pct": round(nulls[c] / total * 100, 2)})
    # string kosong / spasi di awal-akhir (sering lolos dari pengecekan null)
    str_cols = [c for c, t in lf.collect_schema().items() if t == pl.String]
    ws = lf.select([(pl.col(c).str.strip_chars() != pl.col(c)).fill_null(False).sum().alias(c) for c in str_cols]
                   + [(pl.col(c) == "").fill_null(False).sum().alias(c + "__empty") for c in str_cols]).collect()
    check("2-null", name, "tidak ada string kosong / spasi berlebih", sum(ws.row(0)), total)

print("\nKolom yang memang boleh null:")
pl.DataFrame(null_rows)

✅ [train_applications] tidak ada null di kolom non-nullable
✅ [train_applications] tidak ada string kosong / spasi berlebih
✅ [test_applications] tidak ada null di kolom non-nullable
✅ [test_applications] tidak ada string kosong / spasi berlebih
✅ [customer_profiles] tidak ada null di kolom non-nullable
✅ [customer_profiles] tidak ada string kosong / spasi berlebih
✅ [previous_applications] tidak ada null di kolom non-nullable
✅ [previous_applications] tidak ada string kosong / spasi berlebih
✅ [repayment_history] tidak ada null di kolom non-nullable
✅ [repayment_history] tidak ada string kosong / spasi berlebih
✅ [account_behavior] tidak ada null di kolom non-nullable
✅ [account_behavior] tidak ada string kosong / spasi berlebih
✅ [bureau_observations] tidak ada null di kolom non-nullable
✅ [bureau_observations] tidak ada string kosong / spasi berlebih
✅ [ecommerce_transactions] tidak ada null di kolom non-nullable
✅ [ecommerce_transactions] tidak ada string kosong / spasi berlebih

K

table,column,null_count,null_pct
str,str,i64,f64
"""previous_applications""","""approved_amount""",42661,21.12
"""previous_applications""","""scheduled_monthly_installment""",42661,21.12
"""previous_applications""","""annual_interest_rate""",42661,21.12
"""previous_applications""","""loan_id""",42661,21.12
"""repayment_history""","""last_payment_datetime""",31714,2.33
"""bureau_observations""","""score""",43839,5.5
"""bureau_observations""","""utilization""",471615,59.13


## 3. Kunci unik & duplikat

In [6]:
S = "3-keys"
check(S, "train", "application_id unik", n_dup(tr, ["application_id"]), tr.height)
check(S, "train", "customer_id unik (1 nasabah = 1 baris)", n_dup(tr, ["customer_id"]), tr.height)
check(S, "test", "application_id unik", n_dup(te, ["application_id"]), te.height)
check(S, "test", "customer_id unik", n_dup(te, ["customer_id"]), te.height)
check(S, "train∩test", "tidak ada application_id yang sama di train & test",
      tr.join(te, on="application_id", how="semi").height)
check(S, "train∩test", "tidak ada customer_id yang sama di train & test",
      tr.join(te, on="customer_id", how="semi").height)
check(S, "customer_profiles", "customer_id unik", n_dup(cp, ["customer_id"]), cp.height)
check(S, "previous_applications", "application_id unik", n_dup(pa, ["application_id"]), pa.height)
check(S, "previous_applications", "loan_id unik (yang tidak null)",
      n_dup(pa.filter(pl.col("loan_id").is_not_null()), ["loan_id"]))
check(S, "previous_applications", "tidak memuat aplikasi terpilih (train/test)",
      pa.join(app, on="application_id", how="semi").height, pa.height)
check(S, "repayment_history", "installment_id unik", n_dup(rh, ["installment_id"]), rh.height)
check(S, "repayment_history", "(loan_id, due_date) unik", n_dup(rh, ["loan_id", "due_date"]), rh.height)
check(S, "account_behavior", "(customer_id, month) unik", n_dup(ab, ["customer_id", "month"]), ab.height)
check(S, "bureau_observations", "bureau_observation_id unik", n_dup(bo, ["bureau_observation_id"]), bo.height)
check(S, "bureau_observations", "(application_id, bureau_provider) unik",
      n_dup(bo, ["application_id", "bureau_provider"]), bo.height)
check(S, "ecommerce_transactions", "order_item_id unik", n_dup(ec, ["order_item_id"]), nrows(ec))

# Duplikat isi: baris identik di semua kolom selain ID baris-nya sendiri
for name, df, row_id in [("repayment_history", rh, "installment_id"), ("account_behavior", ab, None),
                         ("bureau_observations", bo, "bureau_observation_id"),
                         ("previous_applications", pa, "application_id")]:
    cols = [c for c in (df.collect_schema() if isinstance(df, pl.LazyFrame) else df.schema) if c != row_id]
    check(S, name, "tidak ada baris duplikat isi (selain ID)", n_dup(df, cols), nrows(df), severity="WARN")

# E-commerce sengaja tidak masuk loop di atas: satu order BOLEH berisi dua item identik
# (kategori, qty, harga sama) dengan order_item_id berbeda. Itu baris sah, BUKAN duplikat.
ec_cols = [c for c in ec.collect_schema() if c != "order_item_id"]
same = (ec.group_by(ec_cols).agg(pl.len().alias("n")).filter(pl.col("n") > 1)
          .select(pl.len().alias("groups"), pl.col("n").sum().alias("rows")).collect())
multi = (ec.group_by("order_id").agg(n=pl.len(), nc=pl.col("product_category").n_unique())
           .filter(pl.col("n") > 1).select(pl.len().alias("multi"), (pl.col("nc") < pl.col("n")).sum().alias("repeat_cat")).collect())
print(f"ℹ️  e-commerce: {same['rows'][0]:,} baris berupa item kembar dalam {same['groups'][0]:,} order → sah, JANGAN dihapus.")
print(f"ℹ️  e-commerce: {multi['repeat_cat'][0]:,} dari {multi['multi'][0]:,} order multi-item punya kategori berulang.")

✅ [train] application_id unik
✅ [train] customer_id unik (1 nasabah = 1 baris)
✅ [test] application_id unik
✅ [test] customer_id unik
✅ [train∩test] tidak ada application_id yang sama di train & test
✅ [train∩test] tidak ada customer_id yang sama di train & test
✅ [customer_profiles] customer_id unik
✅ [previous_applications] application_id unik
✅ [previous_applications] loan_id unik (yang tidak null)
✅ [previous_applications] tidak memuat aplikasi terpilih (train/test)
✅ [repayment_history] installment_id unik
✅ [repayment_history] (loan_id, due_date) unik
✅ [account_behavior] (customer_id, month) unik
✅ [bureau_observations] bureau_observation_id unik
✅ [bureau_observations] (application_id, bureau_provider) unik
✅ [ecommerce_transactions] order_item_id unik
✅ [repayment_history] tidak ada baris duplikat isi (selain ID)
✅ [account_behavior] tidak ada baris duplikat isi (selain ID)
✅ [bureau_observations] tidak ada baris duplikat isi (selain ID)
✅ [previous_applications] tidak ada bar

## 4. Domain kategori & rentang angka

In [7]:
S = "4-domain"
DOMAINS = [
    ("train/test", app, "loan_purpose", ["education", "home_improvement", "household", "working_capital", "medical"]),
    ("customer_profiles", cp, "region", ["Region_A", "Region_B", "Region_C", "Region_D", "Region_E"]),
    ("customer_profiles", cp, "employment_type", ["salaried", "self_employed", "variable_income"]),
    ("customer_profiles", cp, "housing_status", ["owned", "rented", "family"]),
    ("previous_applications", pa, "decision", ["approved", "rejected", "cancelled"]),
    ("bureau_observations", bo, "bureau_provider", ["Bureau_A", "Bureau_B", "Bureau_C"]),
    ("ecommerce_transactions", ec, "product_category",
     ["groceries", "fashion", "electronics", "home", "health", "education", "personal_care"]),
    ("ecommerce_transactions", ec, "payment_method", ["COD", "e_wallet", "bank_transfer", "debit_card", "credit_card"]),
    ("ecommerce_transactions", ec, "order_status", ["completed", "cancelled"]),
]
for table, df, col, allowed in DOMAINS:
    check(S, table, f"{col} ∈ nilai di kamus data", nbad(df, ~pl.col(col).is_in(allowed)), nrows(df))

✅ [train/test] loan_purpose ∈ nilai di kamus data
✅ [customer_profiles] region ∈ nilai di kamus data
✅ [customer_profiles] employment_type ∈ nilai di kamus data
✅ [customer_profiles] housing_status ∈ nilai di kamus data
✅ [previous_applications] decision ∈ nilai di kamus data
✅ [bureau_observations] bureau_provider ∈ nilai di kamus data
✅ [ecommerce_transactions] product_category ∈ nilai di kamus data
✅ [ecommerce_transactions] payment_method ∈ nilai di kamus data
✅ [ecommerce_transactions] order_status ∈ nilai di kamus data


In [8]:
RANGES = [
    # (tabel, df, deskripsi, ekspresi PELANGGARAN, severity)
    ("train/test", app, "loan_amount > 0", pl.col("loan_amount") <= 0, "FAIL"),
    ("train/test", app, "term_months == 12 (sesuai kamus data)", pl.col("term_months") != 12, "FAIL"),
    ("train/test", app, "0 < annual_interest_rate < 1", ~pl.col("annual_interest_rate").is_between(0, 1, closed="none"), "FAIL"),
    ("train/test", app, "proposed_monthly_installment > 0", pl.col("proposed_monthly_installment") <= 0, "FAIL"),
    ("customer_profiles", cp, "17 ≤ age ≤ 100", ~pl.col("age").is_between(17, 100), "FAIL"),
    ("customer_profiles", cp, "employment_tenure_months ≥ 0", pl.col("employment_tenure_months") < 0, "FAIL"),
    ("customer_profiles", cp, "masa kerja ≤ (umur − 15) tahun", pl.col("employment_tenure_months") > (pl.col("age") - 15) * 12, "WARN"),
    ("customer_profiles", cp, "declared_monthly_income > 0", pl.col("declared_monthly_income") <= 0, "FAIL"),
    ("customer_profiles", cp, "dependents ≥ 0", pl.col("dependents") < 0, "FAIL"),
    ("previous_applications", pa, "requested_amount > 0", pl.col("requested_amount") <= 0, "FAIL"),
    ("previous_applications", pa, "term_months > 0", pl.col("term_months") <= 0, "FAIL"),
    ("previous_applications", pa, "0 < annual_interest_rate < 1 (jika ada)", ~pl.col("annual_interest_rate").is_between(0, 1, closed="none"), "FAIL"),
    ("repayment_history", rh, "amount_due > 0", pl.col("amount_due") <= 0, "FAIL"),
    ("repayment_history", rh, "amount_paid ≥ 0", pl.col("amount_paid") < 0, "FAIL"),
    ("repayment_history", rh, "days_past_due ≥ 0", pl.col("days_past_due") < 0, "FAIL"),
    ("account_behavior", ab, "saldo (opening/closing/avg/min) ≥ 0",
     pl.min_horizontal("opening_balance", "closing_balance", "average_balance", "minimum_balance") < 0, "WARN"),
    ("account_behavior", ab, "inflows, outflows, income_like_credits ≥ 0",
     pl.min_horizontal("inflows", "outflows", "income_like_credits") < 0, "FAIL"),
    ("account_behavior", ab, "0 ≤ low_balance_days ≤ jumlah hari di bulan itu",
     ~pl.col("low_balance_days").is_between(0, pl.col("month").dt.month_end().dt.day()), "FAIL"),
    ("bureau_observations", bo, "300 ≤ score ≤ 850 (jika ada)", ~pl.col("score").is_between(300, 850), "FAIL"),
    ("bureau_observations", bo, "utilization ≥ 0 (jika ada)", pl.col("utilization") < 0, "FAIL"),
    ("bureau_observations", bo, "utilization ≤ 1 (tidak overlimit)", pl.col("utilization") > 1, "WARN"),
    ("bureau_observations", bo, "hitungan & saldo ≥ 0",
     pl.min_horizontal("active_accounts", "outstanding_balance", "revolving_credit_limit",
                       "revolving_outstanding_balance", "inquiries_last_6m", "delinquent_accounts_last_12m") < 0, "FAIL"),
    ("ecommerce_transactions", ec, "quantity ≥ 1", pl.col("quantity") < 1, "WARN"),
    ("ecommerce_transactions", ec, "item_total > 0", pl.col("item_total") <= 0, "WARN"),
    ("ecommerce_transactions", ec, "order_total ≥ item_total", pl.col("order_total") < pl.col("item_total"), "FAIL"),
]
for table, df, desc, bad_expr, sev in RANGES:
    check(S, table, desc, nbad(df, bad_expr), nrows(df), severity=sev)

✅ [train/test] loan_amount > 0
✅ [train/test] term_months == 12 (sesuai kamus data)
✅ [train/test] 0 < annual_interest_rate < 1
✅ [train/test] proposed_monthly_installment > 0
✅ [customer_profiles] 17 ≤ age ≤ 100
✅ [customer_profiles] employment_tenure_months ≥ 0
✅ [customer_profiles] masa kerja ≤ (umur − 15) tahun
✅ [customer_profiles] declared_monthly_income > 0
✅ [customer_profiles] dependents ≥ 0
✅ [previous_applications] requested_amount > 0
✅ [previous_applications] term_months > 0
✅ [previous_applications] 0 < annual_interest_rate < 1 (jika ada)
✅ [repayment_history] amount_due > 0
✅ [repayment_history] amount_paid ≥ 0
✅ [repayment_history] days_past_due ≥ 0
✅ [account_behavior] saldo (opening/closing/avg/min) ≥ 0
✅ [account_behavior] inflows, outflows, income_like_credits ≥ 0
✅ [account_behavior] 0 ≤ low_balance_days ≤ jumlah hari di bulan itu
✅ [bureau_observations] 300 ≤ score ≤ 850 (jika ada)
✅ [bureau_observations] utilization ≥ 0 (jika ada)
✅ [bureau_observations] utilizat

In [9]:
# Sekilas distribusi kategori (proporsi %)
def vc(df, col):
    out = (df.group_by(col).agg(pl.len().alias("n")))
    out = out.collect() if isinstance(out, pl.LazyFrame) else out
    return out.with_columns((pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct")).sort("n", descending=True)

for table, df, col, _ in DOMAINS:
    d = vc(df, col)
    print(f"{table}.{col}: " + ", ".join(f"{r[col]}={r['pct']}%" for r in d.iter_rows(named=True)))
for col in ["annual_interest_rate", "term_months"]:
    print(f"previous_applications.{col}: ", dict(vc(pa, col).select(col, "pct").iter_rows()))
print("train/test.annual_interest_rate:", dict(vc(app, "annual_interest_rate").select("annual_interest_rate", "pct").iter_rows()))

train/test.loan_purpose: home_improvement=20.17%, education=20.02%, household=20.0%, working_capital=19.97%, medical=19.84%
customer_profiles.region: Region_E=20.25%, Region_D=19.97%, Region_A=19.97%, Region_B=19.94%, Region_C=19.88%
customer_profiles.employment_type: salaried=48.03%, self_employed=29.93%, variable_income=22.04%
customer_profiles.housing_status: rented=43.03%, family=29.04%, owned=27.93%
previous_applications.decision: approved=78.88%, rejected=12.02%, cancelled=9.11%
bureau_observations.bureau_provider: Bureau_A=37.86%, Bureau_C=31.08%, Bureau_B=31.05%
ecommerce_transactions.product_category: groceries=26.58%, fashion=17.71%, personal_care=15.92%, home=13.27%, health=10.62%, education=8.83%, electronics=7.07%
ecommerce_transactions.payment_method: COD=40.58%, e_wallet=23.37%, bank_transfer=14.83%, debit_card=14.73%, credit_card=6.49%
ecommerce_transactions.order_status: completed=93.97%, cancelled=6.03%
previous_applications.annual_interest_rate:  {0.2: 33.19, 0.24: 2

## 5. Integritas relasi antar tabel

```
train/test ─┬─ customer_profiles        (1:1, customer_id)
            ├─ account_behavior         (1:N, customer_id)
            ├─ ecommerce_transactions   (1:N, customer_id)
            ├─ bureau_observations      (1:N, customer_id; application_id → aplikasi terpilih ATAU aplikasi lama)
            └─ previous_applications    (1:N, customer_id) ── repayment_history (1:N, loan_id)
```

In [10]:
S = "5-relasi"
sel_cust = app.select("customer_id")
check(S, "train/test→customer_profiles", "setiap nasabah terpilih punya profil",
      sel_cust.join(cp, on="customer_id", how="anti").height, app.height)
check(S, "customer_profiles→train/test", "setiap profil milik nasabah terpilih",
      cp.join(sel_cust, on="customer_id", how="anti").height, cp.height)
for name, df in [("previous_applications", pa), ("repayment_history", rh), ("account_behavior", ab),
                 ("bureau_observations", bo), ("ecommerce_transactions", ec)]:
    other = df.select("customer_id").unique()
    other = other.collect() if isinstance(other, pl.LazyFrame) else other
    check(S, name, "semua customer_id ada di train/test", other.join(sel_cust, on="customer_id", how="anti").height)

loans = pa.filter(pl.col("loan_id").is_not_null()).select("loan_id", pl.col("customer_id").alias("pa_customer"))
x = rh.join(loans, on="loan_id", how="left")
check(S, "repayment_history→previous_applications", "setiap loan_id ada di aplikasi lama yang approved",
      nbad(x, pl.col("pa_customer").is_null()), rh.height)
check(S, "repayment_history→previous_applications", "customer_id cicilan = customer_id pinjaman",
      nbad(x, pl.col("pa_customer") != pl.col("customer_id")), rh.height)

all_apps = pl.concat([app.select("application_id", "customer_id", pl.lit("selected").alias("app_type")),
                      pa.select("application_id", "customer_id", pl.lit("previous").alias("app_type"))])
bx = bo.join(all_apps.rename({"customer_id": "app_customer"}), on="application_id", how="left")
check(S, "bureau_observations→applications", "setiap application_id ada (terpilih atau lama)",
      nbad(bx, pl.col("app_customer").is_null()), bo.height)
check(S, "bureau_observations→applications", "customer_id laporan = customer_id aplikasi",
      nbad(bx, pl.col("app_customer") != pl.col("customer_id")), bo.height)
check(S, "bureau_observations", "setiap aplikasi terpilih punya ≥1 laporan biro",
      app.join(bo, on="application_id", how="anti").height, app.height, severity="WARN")

check(S, "sample_submission", "ID sama persis dengan test",
      ss.join(te, on="application_id", how="anti").height + te.join(ss, on="application_id", how="anti").height)

print("\nLaporan biro per jenis aplikasi:")
bx.group_by("app_type").agg(pl.len().alias("rows")).with_columns((pl.col("rows") / bo.height * 100).round(1).alias("pct"))

✅ [train/test→customer_profiles] setiap nasabah terpilih punya profil
✅ [customer_profiles→train/test] setiap profil milik nasabah terpilih
✅ [previous_applications] semua customer_id ada di train/test
✅ [repayment_history] semua customer_id ada di train/test
✅ [account_behavior] semua customer_id ada di train/test
✅ [bureau_observations] semua customer_id ada di train/test
✅ [ecommerce_transactions] semua customer_id ada di train/test
✅ [repayment_history→previous_applications] setiap loan_id ada di aplikasi lama yang approved
✅ [repayment_history→previous_applications] customer_id cicilan = customer_id pinjaman
✅ [bureau_observations→applications] setiap application_id ada (terpilih atau lama)
✅ [bureau_observations→applications] customer_id laporan = customer_id aplikasi
✅ [bureau_observations] setiap aplikasi terpilih punya ≥1 laporan biro
✅ [sample_submission] ID sama persis dengan test

Laporan biro per jenis aplikasi:


app_type,rows,pct
str,u32,f64
"""selected""",264088,33.1
"""previous""",533475,66.9


## 6. Konsistensi aturan bisnis (semantik kolom)

Bagian ini sekaligus **menguji pemahaman kita tentang arti setiap kolom**. Kalau sebuah aturan 100% terpenuhi, kita tahu persis bagaimana kolom itu dibentuk, dan itu berguna untuk feature engineering.

### 6a. Aplikasi terpilih & aplikasi lama

In [11]:
S = "6-konsistensi"

def annuity(P, annual_rate, n):
    r = annual_rate / 12
    return P * r / (1 - (1 + r) ** (-n))

# Hipotesis: cicilan = rumus anuitas, DIBULATKAN KE ATAS ke rupiah utuh
d = app.with_columns(gap=pl.col("proposed_monthly_installment")
                     - annuity(pl.col("loan_amount"), pl.col("annual_interest_rate"), pl.col("term_months")))
check(S, "train/test", "proposed_monthly_installment = ceil(anuitas)",
      nbad(d, (pl.col("gap") < -1e-6) | (pl.col("gap") >= 1 + 1e-6)), app.height)

appr = pa.filter(pl.col("decision") == "approved")
d = appr.with_columns(gap=pl.col("scheduled_monthly_installment")
                      - annuity(pl.col("approved_amount"), pl.col("annual_interest_rate"), pl.col("term_months")))
check(S, "previous_applications", "scheduled_monthly_installment = ceil(anuitas)",
      nbad(d, (pl.col("gap") < -1e-6) | (pl.col("gap") >= 1 + 1e-6)), appr.height)

is_appr = pl.col("decision") == "approved"
for c in ["loan_id", "approved_amount", "annual_interest_rate", "scheduled_monthly_installment"]:
    check(S, "previous_applications", f"{c} terisi ⇔ decision = approved",
          nbad(pa, pl.col(c).is_not_null() != is_appr), pa.height)
check(S, "previous_applications", "approved_amount ≤ requested_amount",
      nbad(appr, pl.col("approved_amount") > pl.col("requested_amount")), appr.height)
check(S, "previous_applications", "application_datetime ≤ decision_datetime",
      nbad(pa, pl.col("application_datetime") > pl.col("decision_datetime")), pa.height)
check(S, "train/test", "application_datetime ≤ decision_datetime",
      nbad(app, pl.col("application_datetime") > pl.col("decision_datetime")), app.height)

cut_amt = appr.select((pl.col("approved_amount") < pl.col("requested_amount")).mean()).item()
print(f"\nℹ️  {cut_amt:.1%} pinjaman lama disetujui dengan nominal LEBIH KECIL dari yang diminta → kandidat fitur.")

✅ [train/test] proposed_monthly_installment = ceil(anuitas)
✅ [previous_applications] scheduled_monthly_installment = ceil(anuitas)
✅ [previous_applications] loan_id terisi ⇔ decision = approved
✅ [previous_applications] approved_amount terisi ⇔ decision = approved
✅ [previous_applications] annual_interest_rate terisi ⇔ decision = approved
✅ [previous_applications] scheduled_monthly_installment terisi ⇔ decision = approved
✅ [previous_applications] approved_amount ≤ requested_amount
✅ [previous_applications] application_datetime ≤ decision_datetime
✅ [train/test] application_datetime ≤ decision_datetime

ℹ️  27.9% pinjaman lama disetujui dengan nominal LEBIH KECIL dari yang diminta → kandidat fitur.


### 6b. Repayment history

Hipotesis semantik yang diuji:
- **Cicilan lunas** (`amount_paid == amount_due`) → `days_past_due = 0`, dan `max_days_past_due` = seberapa telat dulu dibayar.
- **Cicilan belum lunas** → `days_past_due` = jumlah hari dari `due_date` sampai **cutoff** (snapshot saat keputusan), dan `max_days_past_due = days_past_due`.
- `last_payment_datetime` kosong ⇔ belum ada pembayaran sama sekali.

In [12]:
loan_info = appr.select("loan_id", "term_months", "scheduled_monthly_installment",
                        pl.col("decision_datetime").alias("loan_decision"))
r = (rh.join(CUT.select("customer_id", "cutoff"), on="customer_id", how="left")
       .join(loan_info, on="loan_id", how="left")
       .with_columns(inst_no=pl.col("due_date").rank("ordinal").over("loan_id"),
                     paid_full=pl.col("amount_paid") >= pl.col("amount_due"),
                     age_days=days_between(pl.col("cutoff"), pl.col("due_date"))))
N = r.height
check(S, "repayment_history", "tidak ada kelebihan bayar (amount_paid ≤ amount_due)", nbad(r, pl.col("amount_paid") > pl.col("amount_due")), N)
check(S, "repayment_history", "last_payment_datetime kosong ⇔ amount_paid = 0",
      nbad(r, pl.col("last_payment_datetime").is_null() != (pl.col("amount_paid") == 0)), N)
check(S, "repayment_history", "cicilan lunas → days_past_due = 0", nbad(r, pl.col("paid_full") & (pl.col("days_past_due") != 0)), N)
check(S, "repayment_history", "cicilan belum lunas → days_past_due = hari sejak due_date s.d. cutoff",
      nbad(r, ~pl.col("paid_full") & (pl.col("days_past_due") != pl.col("age_days"))), N)
check(S, "repayment_history", "days_past_due ≤ max_days_past_due", nbad(r, pl.col("days_past_due") > pl.col("max_days_past_due")), N)
check(S, "repayment_history", "cicilan belum lunas → max_dpd = dpd",
      nbad(r, ~pl.col("paid_full") & (pl.col("max_days_past_due") != pl.col("days_past_due"))), N)
check(S, "repayment_history", "jumlah cicilan per pinjaman ≤ tenor", nbad(r, pl.col("inst_no") > pl.col("term_months")), N)
check(S, "repayment_history", "amount_due = cicilan terjadwal (kecuali cicilan terakhir)",
      nbad(r, (pl.col("inst_no") < pl.col("term_months")) & (pl.col("amount_due") != pl.col("scheduled_monthly_installment"))), N)

# Hipotesis: cicilan ke-k jatuh tempo tepat k bulan kalender setelah keputusan pinjaman (jam yang sama)
expected_due = pl.col("loan_decision").dt.offset_by(pl.format("{}mo", pl.col("inst_no")))
check(S, "repayment_history", "due_date cicilan ke-k = keputusan pinjaman + k bulan (persis)",
      nbad(r, pl.col("due_date") != expected_due), N)

paid = r.filter(pl.col("paid_full")).with_columns(late=days_between(pl.col("last_payment_datetime"), pl.col("due_date")))
print("\nℹ️  Status cicilan:")
print(r.select(lunas=pl.col("paid_full").mean(), bayar_sebagian=((pl.col("amount_paid") > 0) & ~pl.col("paid_full")).mean(),
               belum_bayar=(pl.col("amount_paid") == 0).mean()).with_columns(pl.all().round(4)))
print(f"ℹ️  Cicilan lunas yang dibayar SEBELUM jatuh tempo: {(paid['late'] < 0).mean():.2%}")
print(f"ℹ️  Cicilan lunas: |max_dpd − hari keterlambatan| ≤ 1 hari pada {((paid['max_days_past_due'] - paid['late']).abs() <= 1).mean():.2%} baris")
print(f"ℹ️  Pinjaman lama yang masih berjalan saat cutoff (cicilan tercatat < tenor): "
      f"{r.group_by('loan_id').agg(pl.len() < pl.col('term_months').first()).select(pl.col('len').mean()).item():.1%}")

✅ [repayment_history] tidak ada kelebihan bayar (amount_paid ≤ amount_due)
✅ [repayment_history] last_payment_datetime kosong ⇔ amount_paid = 0
✅ [repayment_history] cicilan lunas → days_past_due = 0
✅ [repayment_history] cicilan belum lunas → days_past_due = hari sejak due_date s.d. cutoff
✅ [repayment_history] days_past_due ≤ max_days_past_due
✅ [repayment_history] cicilan belum lunas → max_dpd = dpd
✅ [repayment_history] jumlah cicilan per pinjaman ≤ tenor
✅ [repayment_history] amount_due = cicilan terjadwal (kecuali cicilan terakhir)
✅ [repayment_history] due_date cicilan ke-k = keputusan pinjaman + k bulan (persis)

ℹ️  Status cicilan:
┌────────┬────────────────┬─────────────┐
│ lunas  ┆ bayar_sebagian ┆ belum_bayar │
│ ---    ┆ ---            ┆ ---         │
│ f64    ┆ f64            ┆ f64         │
╞════════╪════════════════╪═════════════╡
│ 0.9568 ┆ 0.0199         ┆ 0.0233      │
└────────┴────────────────┴─────────────┘
ℹ️  Cicilan lunas yang dibayar SEBELUM jatuh tempo: 0.00%

### 6c. Account behavior

In [13]:
a = (ab.sort("customer_id", "month")
       .with_columns(prev_close=pl.col("closing_balance").shift(1).over("customer_id"),
                     prev_month=pl.col("month").shift(1).over("customer_id")))
N = a.height
check(S, "account_behavior", "closing = opening + inflows − outflows",
      nbad(a, pl.col("opening_balance") + pl.col("inflows") - pl.col("outflows") != pl.col("closing_balance")), N)
check(S, "account_behavior", "opening bulan ini = closing bulan lalu",
      nbad(a, pl.col("prev_close").is_not_null() & (pl.col("opening_balance") != pl.col("prev_close"))), N)
check(S, "account_behavior", "bulan berurutan tanpa lompatan",
      nbad(a, pl.col("prev_month").is_not_null() & (pl.col("month") != pl.col("prev_month").dt.offset_by("1mo"))), N, severity="WARN")
check(S, "account_behavior", "minimum_balance ≤ average_balance", nbad(a, pl.col("minimum_balance") > pl.col("average_balance")), N)
check(S, "account_behavior", "minimum_balance ≤ closing_balance", nbad(a, pl.col("minimum_balance") > pl.col("closing_balance")), N)
check(S, "account_behavior", "income_like_credits ≤ inflows", nbad(a, pl.col("income_like_credits") > pl.col("inflows")), N)
check(S, "account_behavior", "low_balance_days > 0 ⇔ minimum_balance < 250.000",
      nbad(a, (pl.col("low_balance_days") > 0) != (pl.col("minimum_balance") < 250_000)), N)
check(S, "account_behavior", "month selalu tanggal 1", nbad(a, pl.col("month").dt.day() != 1), N)

✅ [account_behavior] closing = opening + inflows − outflows
✅ [account_behavior] opening bulan ini = closing bulan lalu
✅ [account_behavior] bulan berurutan tanpa lompatan
✅ [account_behavior] minimum_balance ≤ average_balance
✅ [account_behavior] minimum_balance ≤ closing_balance
✅ [account_behavior] income_like_credits ≤ inflows
✅ [account_behavior] low_balance_days > 0 ⇔ minimum_balance < 250.000
✅ [account_behavior] month selalu tanggal 1


### 6d. Bureau observations

In [14]:
N = bo.height
check(S, "bureau_observations", "utilization = revolving_outstanding / revolving_limit",
      nbad(bo, (pl.col("utilization") - pl.col("revolving_outstanding_balance") / pl.col("revolving_credit_limit")).abs() > 1e-6), N)
check(S, "bureau_observations", "utilization kosong ⇔ revolving_credit_limit ≤ 0",
      nbad(bo, pl.col("utilization").is_null() != (pl.col("revolving_credit_limit") <= 0)), N)
check(S, "bureau_observations", "revolving_outstanding ≤ outstanding_balance",
      nbad(bo, pl.col("revolving_outstanding_balance") > pl.col("outstanding_balance")), N)
check(S, "bureau_observations", "active_accounts = 0 → outstanding_balance = 0",
      nbad(bo, (pl.col("active_accounts") == 0) & (pl.col("outstanding_balance") > 0)), N)
check(S, "bureau_observations", "report_datetime ≤ available_datetime",
      nbad(bo, pl.col("report_datetime") > pl.col("available_datetime")), N)

print("\nℹ️  Skor kosong per provider & jeda report → available (hari):")
(bo.group_by("bureau_provider")
   .agg(rows=pl.len(), score_null_pct=(pl.col("score").is_null().mean() * 100).round(2),
        util_null_pct=(pl.col("utilization").is_null().mean() * 100).round(2),
        lag_days_median=days_between(pl.col("available_datetime"), pl.col("report_datetime")).median(),
        lag_days_max=days_between(pl.col("available_datetime"), pl.col("report_datetime")).max(),
        score_mean=pl.col("score").mean().round(1))
   .sort("bureau_provider"))

✅ [bureau_observations] utilization = revolving_outstanding / revolving_limit
✅ [bureau_observations] utilization kosong ⇔ revolving_credit_limit ≤ 0
✅ [bureau_observations] revolving_outstanding ≤ outstanding_balance
✅ [bureau_observations] active_accounts = 0 → outstanding_balance = 0
✅ [bureau_observations] report_datetime ≤ available_datetime

ℹ️  Skor kosong per provider & jeda report → available (hari):


bureau_provider,rows,score_null_pct,util_null_pct,lag_days_median,lag_days_max,score_mean
str,u32,f64,f64,f64,i64,f64
"""Bureau_A""",301984,5.24,57.14,60.0,120,669.0
"""Bureau_B""",247657,5.56,59.1,60.0,120,669.0
"""Bureau_C""",247922,5.75,61.58,60.0,120,669.0


### 6e. E-commerce

In [15]:
orders = (ec.group_by("order_id")
            .agg(n_items=pl.len(), sum_items=pl.col("item_total").sum(),
                 **{f"nu_{c}": pl.col(c).n_unique() for c in
                    ["customer_id", "transaction_datetime", "payment_method", "order_status", "order_total"]},
                 order_total=pl.col("order_total").first(), status=pl.col("order_status").first())
            .collect())
N = orders.height
check(S, "ecommerce_transactions", "Σ item_total = order_total (per order)", nbad(orders, pl.col("sum_items") != pl.col("order_total")), N)
for c in ["customer_id", "transaction_datetime", "payment_method", "order_status", "order_total"]:
    check(S, "ecommerce_transactions", f"{c} konstan dalam satu order", nbad(orders, pl.col(f"nu_{c}") > 1), N)

print(f"\nℹ️  {N:,} order unik | rata-rata {orders['n_items'].mean():.2f} item/order | "
      f"cancelled {(orders['status'] == 'cancelled').mean():.2%}")

✅ [ecommerce_transactions] Σ item_total = order_total (per order)
✅ [ecommerce_transactions] customer_id konstan dalam satu order
✅ [ecommerce_transactions] transaction_datetime konstan dalam satu order
✅ [ecommerce_transactions] payment_method konstan dalam satu order
✅ [ecommerce_transactions] order_status konstan dalam satu order
✅ [ecommerce_transactions] order_total konstan dalam satu order

ℹ️  2,986,855 order unik | rata-rata 1.46 item/order | cancelled 6.03%


## 7. Kepatuhan cutoff (kebocoran data)

Aturan lomba: hanya informasi yang tersedia **sebelum `decision_datetime`** yang boleh dipakai. Di sini kita cek apakah ada baris "dari masa depan" di setiap tabel riwayat.

In [16]:
S = "7-cutoff"
def with_cut(df):
    return df.join(CUT.lazy() if isinstance(df, pl.LazyFrame) else CUT, on="customer_id", how="left")

p = with_cut(pa)
check(S, "previous_applications", "application_datetime < cutoff", nbad(p, pl.col("application_datetime") >= pl.col("cutoff")), pa.height)
check(S, "previous_applications", "decision_datetime < cutoff (keputusan lama sudah diketahui)",
      nbad(p, pl.col("decision_datetime") >= pl.col("cutoff")), pa.height)
r2 = with_cut(rh)
check(S, "repayment_history", "due_date < cutoff", nbad(r2, pl.col("due_date") >= pl.col("cutoff")), rh.height)
check(S, "repayment_history", "last_payment_datetime < cutoff", nbad(r2, pl.col("last_payment_datetime") >= pl.col("cutoff")), rh.height)
a2 = with_cut(ab)
check(S, "account_behavior", "bulan sudah lengkap sebelum cutoff (awal bulan berikutnya ≤ cutoff)",
      nbad(a2, pl.col("month").dt.offset_by("1mo").cast(pl.Datetime("us")) > pl.col("cutoff")), ab.height)
b2 = with_cut(bo)
check(S, "bureau_observations", "available_datetime ≤ cutoff", nbad(b2, pl.col("available_datetime") > pl.col("cutoff")), bo.height)
check(S, "bureau_observations", "report_datetime ≤ cutoff", nbad(b2, pl.col("report_datetime") > pl.col("cutoff")), bo.height)
e2 = with_cut(ec)
check(S, "ecommerce_transactions", "transaction_datetime < cutoff", nbad(e2, pl.col("transaction_datetime") >= pl.col("cutoff")), nrows(ec))

# Pinjaman lama approved tanpa satu pun cicilan: wajar HANYA jika cicilan pertamanya belum jatuh tempo saat cutoff
no_inst = with_cut(appr.join(rh.select("loan_id").unique(), on="loan_id", how="anti"))
check(S, "previous_applications", "loan approved tanpa cicilan hanya jika due pertama ≥ cutoff",
      nbad(no_inst, pl.col("decision_datetime").dt.offset_by("1mo") < pl.col("cutoff")), no_inst.height)
print(f"\nℹ️  {no_inst.height:,} pinjaman lama approved belum punya cicilan jatuh tempo (disetujui < 1 bulan sebelum cutoff).")

✅ [previous_applications] application_datetime < cutoff
✅ [previous_applications] decision_datetime < cutoff (keputusan lama sudah diketahui)
✅ [repayment_history] due_date < cutoff
✅ [repayment_history] last_payment_datetime < cutoff
✅ [account_behavior] bulan sudah lengkap sebelum cutoff (awal bulan berikutnya ≤ cutoff)
✅ [bureau_observations] available_datetime ≤ cutoff
✅ [bureau_observations] report_datetime ≤ cutoff
✅ [ecommerce_transactions] transaction_datetime < cutoff
✅ [previous_applications] loan approved tanpa cicilan hanya jika due pertama ≥ cutoff

ℹ️  3,422 pinjaman lama approved belum punya cicilan jatuh tempo (disetujui < 1 bulan sebelum cutoff).


In [17]:
# Seberapa "segar" informasi saat keputusan?
fresh = pl.concat([
    b2.group_by("customer_id").agg(v=days_between(pl.col("cutoff"), pl.col("report_datetime")).min())
      .select(pl.lit("bureau: hari sejak laporan terbaru").alias("info"), "v"),
    a2.group_by("customer_id").agg(v=((pl.col("cutoff").dt.year() - pl.col("month").dt.year()) * 12
                                      + pl.col("cutoff").dt.month() - pl.col("month").dt.month()).min())
      .select(pl.lit("rekening: jarak bulan terakhir (bulan)").alias("info"), "v"),
    a2.group_by("customer_id").agg(v=pl.len()).select(pl.lit("rekening: jumlah bulan tersedia").alias("info"), "v"),
    r2.group_by("customer_id").agg(v=days_between(pl.col("cutoff"), pl.col("due_date")).min())
      .select(pl.lit("repayment: hari sejak due_date terakhir").alias("info"), "v"),
    e2.group_by("customer_id").agg(v=days_between(pl.col("cutoff"), pl.col("transaction_datetime")).min()).collect()
      .select(pl.lit("e-commerce: hari sejak transaksi terakhir").alias("info"), "v"),
    app.select(pl.lit("aplikasi: jam dari pengajuan ke keputusan").alias("info"),
               ((pl.col("decision_datetime") - pl.col("application_datetime")).dt.total_seconds() // 3600).cast(pl.Int64).alias("v")),
], how="vertical_relaxed")
(fresh.group_by("info", maintain_order=True)
      .agg(n=pl.len(), min=pl.col("v").min(), p25=pl.col("v").quantile(0.25), median=pl.col("v").median(),
           p75=pl.col("v").quantile(0.75), max=pl.col("v").max()))

info,n,min,p25,median,p75,max
str,u32,i64,f64,f64,f64,i64
"""bureau: hari sejak laporan terbaru""",100000,0,12.0,28.0,50.0,122
"""rekening: jarak bulan terakhir (bulan)""",70000,1,1.0,1.0,1.0,1
"""rekening: jumlah bulan tersedia""",70000,8,11.0,15.0,20.0,23
"""repayment: hari sejak due_date terakhir""",66157,0,6.0,15.0,29.0,547
"""e-commerce: hari sejak transaksi terakhir""",100000,1,3.0,11.0,30.0,357
"""aplikasi: jam dari pengajuan ke keputusan""",100000,0,1.0,9.0,26.0,119


## 8. Cakupan tabel per split

Pola "punya / tidak punya data" harus sama di train dan test. Kalau beda, model akan belajar pola missing yang tidak berlaku di test.

In [18]:
S = "8-cakupan"
def custset(df):
    out = df.select("customer_id").unique()
    return out.collect() if isinstance(out, pl.LazyFrame) else out

cov = app.select("customer_id", "split")
for name, df in [("prev_apps", pa), ("approved_loans", appr), ("repayment", rh),
                 ("account", ab), ("bureau", bo), ("ecommerce", ec)]:
    cov = cov.join(custset(df).with_columns(pl.lit(True).alias(f"has_{name}")), on="customer_id", how="left")
cov = cov.with_columns(pl.col("^has_.*$").fill_null(False))
cov = cov.join(tr.select("customer_id", "target"), on="customer_id", how="left")

has_cols = [c for c in cov.columns if c.startswith("has_")]
coverage = (cov.group_by("split").agg([(pl.col(c).mean() * 100).round(2).alias(c) for c in has_cols]).sort("split", descending=True))
display(coverage)
for c in has_cols:
    t = coverage.filter(pl.col("split") == "train")[c].item()
    s = coverage.filter(pl.col("split") == "test")[c].item()
    check(S, "train vs test", f"cakupan {c} beda < 1 poin persen", int(abs(t - s) >= 1), severity="WARN",
          note=f"train {t}% vs test {s}%")

print("Target rate (train) menurut ketersediaan data:")
(pl.concat([cov.filter(pl.col("split") == "train").group_by(c).agg(n=pl.len(), target_rate=pl.col("target").mean().round(4))
             .rename({c: "has_data"}).with_columns(pl.lit(c).alias("table")) for c in has_cols])
   .select("table", "has_data", "n", "target_rate").sort("table", "has_data"))

split,has_prev_apps,has_approved_loans,has_repayment,has_account,has_bureau,has_ecommerce
str,f64,f64,f64,f64,f64,f64
"""train""",70.0,66.52,66.16,70.0,100.0,100.0
"""test""",70.0,66.51,66.15,70.0,100.0,100.0


✅ [train vs test] cakupan has_prev_apps beda < 1 poin persen
✅ [train vs test] cakupan has_approved_loans beda < 1 poin persen
✅ [train vs test] cakupan has_repayment beda < 1 poin persen
✅ [train vs test] cakupan has_account beda < 1 poin persen
✅ [train vs test] cakupan has_bureau beda < 1 poin persen
✅ [train vs test] cakupan has_ecommerce beda < 1 poin persen
Target rate (train) menurut ketersediaan data:


table,has_data,n,target_rate
str,bool,u32,f64
"""has_account""",false,21000,0.1172
"""has_account""",true,49000,0.1438
"""has_approved_loans""",false,23435,0.121
"""has_approved_loans""",true,46565,0.1433
"""has_bureau""",true,70000,0.1358
"""has_ecommerce""",true,70000,0.1358
"""has_prev_apps""",false,21000,0.1172
"""has_prev_apps""",true,49000,0.1438
"""has_repayment""",false,23688,0.1215


## 9. Train vs test: apakah distribusinya sama?

Untuk setiap kolom (plus beberapa ringkasan per nasabah dari tabel riwayat):
- **KS** (numerik): selisih maksimum distribusi kumulatif train vs test. Dengan 70k vs 30k baris, KS ≈ 0,01 sudah "signifikan" secara statistik, jadi yang diberi ⚠️ hanya yang **juga** besar secara praktis (KS > 0,02).
- **PSI** (kategori): < 0,1 dianggap stabil.

In [19]:
S = "9-drift"
cust_feats = (
    app.select("customer_id", "split",
               "loan_amount", "annual_interest_rate", "proposed_monthly_installment", "loan_purpose",
               lag_hours=(pl.col("decision_datetime") - pl.col("application_datetime")).dt.total_seconds() / 3600,
               decision_month=pl.col("decision_datetime").dt.month(),
               decision_hour=pl.col("decision_datetime").dt.hour())
    .join(cp, on="customer_id", how="left")
    .join(pa.group_by("customer_id").agg(n_prev_apps=pl.len(), n_rejected=(pl.col("decision") == "rejected").sum()),
          on="customer_id", how="left")
    .join(rh.group_by("customer_id").agg(n_installments=pl.len(), max_dpd_hist=pl.col("max_days_past_due").max()),
          on="customer_id", how="left")
    .join(ab.group_by("customer_id").agg(n_months=pl.len(), avg_balance_mean=pl.col("average_balance").mean()),
          on="customer_id", how="left")
    .join(bo.join(app.select("application_id"), on="application_id", how="semi")
            .group_by("customer_id").agg(sel_score_mean=pl.col("score").mean(), sel_n_reports=pl.len()),
          on="customer_id", how="left")
    .join(ec.group_by("customer_id").agg(n_orders=pl.col("order_id").n_unique(), n_items=pl.len()).collect(),
          on="customer_id", how="left")
)

def psi(a, b, eps=1e-6):
    cats = sorted(set(a) | set(b))
    pa_ = np.array([(a == c).mean() for c in cats]) + eps
    pb_ = np.array([(b == c).mean() for c in cats]) + eps
    return float(np.sum((pa_ - pb_) * np.log(pa_ / pb_)))

CAT = ["loan_purpose", "region", "employment_type", "housing_status", "annual_interest_rate", "dependents", "decision_month"]
NUM = [c for c, t in cust_feats.schema.items() if t.is_numeric() and c not in CAT]
trf, tef = cust_feats.filter(pl.col("split") == "train"), cust_feats.filter(pl.col("split") == "test")
rows = []
for c in NUM:
    a_, b_ = trf[c].drop_nulls().to_numpy(), tef[c].drop_nulls().to_numpy()
    ks = ks_2samp(a_, b_)
    rows.append(dict(feature=c, type="num", ks=round(ks.statistic, 4), p_value=round(ks.pvalue, 4), psi=None,
                     null_train=round(trf[c].is_null().mean(), 4), null_test=round(tef[c].is_null().mean(), 4)))
for c in CAT:
    a_, b_ = trf[c].cast(pl.String).to_numpy(), tef[c].cast(pl.String).to_numpy()
    rows.append(dict(feature=c, type="cat", ks=None, p_value=None, psi=round(psi(a_, b_), 5),
                     null_train=0.0, null_test=0.0))
drift = pl.DataFrame(rows).sort("ks", descending=True, nulls_last=True)
for row in drift.iter_rows(named=True):
    bad = (row["ks"] is not None and row["ks"] > 0.02 and row["p_value"] < 0.01) or (row["psi"] is not None and row["psi"] > 0.1)
    if bad:
        check(S, "train vs test", f"distribusi {row['feature']} stabil", 1, severity="WARN",
              note=f"KS={row['ks']} PSI={row['psi']}")
check(S, "train vs test", f"tidak ada drift berarti pada {drift.height} fitur yang dicek",
      int(sum(1 for r_ in CHECKS if r_['section'] == S and r_['status'] != 'PASS')), severity="WARN")
drift

✅ [train vs test] tidak ada drift berarti pada 24 fitur yang dicek


feature,type,ks,p_value,psi,null_train,null_test
str,str,f64,f64,f64,f64,f64
"""avg_balance_mean""","""num""",0.011,0.0571,null,0.3,0.3
"""employment_tenure_months""","""num""",0.0099,0.033,null,0.0,0.0
"""age""","""num""",0.0095,0.0463,null,0.0,0.0
"""n_items""","""num""",0.0085,0.096,null,0.0,0.0
"""lag_hours""","""num""",0.0084,0.102,null,0.0,0.0
"""n_orders""","""num""",0.0063,0.3763,null,0.0,0.0
"""n_months""","""num""",0.006,0.6573,null,0.3,0.3
"""declared_monthly_income""","""num""",0.0057,0.4942,null,0.0,0.0
"""sel_score_mean""","""num""",0.0047,0.7527,null,0.0025,0.0023


## 10. Outlier & ekor distribusi

Bukan PASS/FAIL. Tujuannya mengetahui kolom mana yang **sangat menceng** (rasio max/p99 besar) sehingga perlu `log1p`, *clipping*, atau fitur rasio. Model berbasis pohon (LightGBM dkk.) relatif tahan outlier, tapi agregasi seperti `mean` / `sum` tetap bisa terdistorsi.

In [20]:
def tail_profile(name, df, cols):
    lf = df.lazy() if isinstance(df, pl.DataFrame) else df
    q = lf.select([pl.struct(
        min=pl.col(c).min().cast(pl.Float64), p1=pl.col(c).quantile(0.01), p50=pl.col(c).median(),
        p99=pl.col(c).quantile(0.99), max=pl.col(c).max().cast(pl.Float64), zero_pct=(pl.col(c) == 0).mean() * 100
    ).alias(c) for c in cols]).collect()
    out = []
    for c in cols:
        s = q[c][0]
        out.append(dict(table=name, column=c, **{k: round(v, 2) if v is not None else None for k, v in s.items()},
                        max_over_p99=round(s["max"] / s["p99"], 2) if s["p99"] else None))
    return out

tails = []
tails += tail_profile("train/test", app, ["loan_amount", "proposed_monthly_installment"])
tails += tail_profile("customer_profiles", cp, ["age", "employment_tenure_months", "declared_monthly_income", "dependents"])
tails += tail_profile("previous_applications", pa, ["requested_amount", "approved_amount"])
tails += tail_profile("repayment_history", rh, ["amount_due", "amount_paid", "days_past_due", "max_days_past_due"])
tails += tail_profile("account_behavior", ab, ["opening_balance", "closing_balance", "inflows", "outflows",
                                               "income_like_credits", "average_balance", "minimum_balance", "low_balance_days"])
tails += tail_profile("bureau_observations", bo, ["score", "active_accounts", "outstanding_balance", "revolving_credit_limit",
                                                  "revolving_outstanding_balance", "utilization", "inquiries_last_6m",
                                                  "delinquent_accounts_last_12m"])
tails += tail_profile("ecommerce_transactions", ec, ["quantity", "item_total", "order_total"])
pl.DataFrame(tails)

table,column,min,p1,p50,p99,max,zero_pct,max_over_p99
str,str,f64,f64,f64,f64,f64,f64,f64
"""train/test""","""loan_amount""",1e6,3.5e6,1.9e7,8.95e7,2.645e8,0.0,2.96
"""train/test""","""proposed_monthly_installment""",90731.0,321503.0,1.769252e6,8.396893e6,2.5011014e7,0.0,2.98
"""customer_profiles""","""age""",23.0,23.0,42.0,60.0,60.0,0.0,1.0
"""customer_profiles""","""employment_tenure_months""",3.0,6.0,44.0,160.0,297.0,0.0,1.86
"""customer_profiles""","""declared_monthly_income""",490493.0,1.703523e6,7.775115e6,3.136308e7,9.1954431e7,0.0,2.93
"""customer_profiles""","""dependents""",0.0,0.0,2.0,4.0,4.0,20.02,1.0
"""previous_applications""","""requested_amount""",1e6,3.6e6,1.93e7,8.68e7,2.724e8,0.0,3.14
"""previous_applications""","""approved_amount""",800000.0,3.4e6,1.83e7,8.43e7,2.724e8,0.0,3.23
"""repayment_history""","""amount_due""",58287.0,257712.0,1.621696e6,9.99119e6,3.7642447e7,0.0,3.77


## 11. Ringkasan audit

In [21]:
res = pl.DataFrame(CHECKS, infer_schema_length=None)
res.write_csv(REPORT_DIR / "audit_results.csv")
summary = res.group_by("status").agg(pl.len().alias("n")).sort("status")
print(f"Total pengecekan: {res.height}")
for s_, n_ in summary.iter_rows():
    print(f"  {ICON[s_]} {s_}: {n_}")
print("\nPer bagian:")
display(res.group_by("section").agg(n=pl.len(), pass_=(pl.col("status") == "PASS").sum(),
                                    warn=(pl.col("status") == "WARN").sum(), fail=(pl.col("status") == "FAIL").sum())
           .sort("section"))
print("Pengecekan yang TIDAK PASS:")
res.filter(pl.col("status") != "PASS")

Total pengecekan: 137
  ✅ PASS: 137

Per bagian:


section,n,pass_,warn,fail
str,u32,u32,u32,u32
"""1-overview""",1,1,0,0
"""2-null""",16,16,0,0
"""3-keys""",20,20,0,0
"""4-domain""",34,34,0,0
"""5-relasi""",13,13,0,0
"""6-konsistensi""",37,37,0,0
"""7-cutoff""",9,9,0,0
"""8-cakupan""",6,6,0,0
"""9-drift""",1,1,0,0


Pengecekan yang TIDAK PASS:


section,table,check,status,n_bad,n_total,pct_bad,note
str,str,str,str,i64,i64,f64,str


## 12. Temuan kunci & implikasinya untuk tahap berikutnya

*(Ditulis dari hasil run pada data kompetisi + e-commerce hasil sampling.)*

**1. Data sangat bersih.** Tidak ada null tak terduga, duplikat kunci, relasi putus, atau nilai di luar domain. Semua aturan bisnis yang diuji terpenuhi 100%. → **Tidak perlu tahap cleaning**; waktu sebaiknya dipakai untuk feature engineering.

**2. Cutoff sudah ditegakkan panitia di semua tabel.** Tidak ada satu baris pun dari masa depan. → Risiko kebocoran tinggal di **cara menghitung fitur**: semua jendela waktu ("3 bulan terakhir", "hari sejak …") wajib relatif terhadap `decision_datetime` masing-masing nasabah, dan target encoding harus out-of-fold.

**3. Semantik kolom yang sekarang terkonfirmasi:**
- **Cicilan = ceil(anuitas)** dari pokok, bunga, tenor. Harga pinjaman sepenuhnya ditentukan oleh `annual_interest_rate` (keputusan risiko lender) + nominal.
- **Repayment adalah snapshot saat cutoff.** Cicilan lunas → `days_past_due = 0` dan `max_days_past_due` = berapa hari telat saat dibayar. Cicilan belum lunas → `days_past_due` = hari sejak jatuh tempo sampai cutoff (**tunggakan aktif**). Cicilan ke-k jatuh tempo tepat k bulan setelah keputusan. Tidak ada bayar lebih awal / lebih bayar.
- **~48% pinjaman lama masih berjalan**, dan 3.422 pinjaman baru disetujui < 1 bulan sebelum cutoff (belum ada cicilan). → Beban cicilan berjalan harus dihitung dari `previous_applications` (jadwal), bukan hanya dari baris repayment.
- **Rekening**: identitas saldo sempurna, bulan berurutan, bulan terakhir selalu = bulan sebelum keputusan; panjang riwayat 8–23 bulan → fitur jendela harus tahan terhadap panjang riwayat yang berbeda.
- **Biro**: `utilization` kosong ⇔ tidak punya limit revolving (59% laporan) → null yang informatif, jangan diisi 0. Laporan terbaru berumur 0–122 hari saat keputusan → fitur *staleness*. 2/3 laporan menempel ke aplikasi lama → bisa dibuat fitur tren.
- **E-commerce**: item kembar dalam satu order itu sah. Agregasi nilai belanja di level order wajib dedup `order_id`, dan order cancelled bukan belanja.

**4. Dua populasi nasabah.** 30% nasabah tidak punya riwayat internal (pengajuan lama + rekening), proporsinya identik di train & test, dan default rate-nya lebih rendah (11,7% vs 14,4%). Untuk mereka hanya ada profil, biro, dan e-commerce.

**5. Train ≈ test.** 24 fitur yang dicek: semua KS < 0,012 dan PSI < 0,001. → **StratifiedKFold** aman, dan skor CV seharusnya mencerminkan leaderboard (selain noise KS-nya sendiri).

**6. Outlier wajar.** Kolom uang menceng (max/p99 ≈ 3–7×) tapi plausibel → cukup `log1p` / rasio, tidak perlu dibuang. `days_past_due` = 0 di ~96% baris → fitur hitungan/biner ("pernah ≥30/60/90 hari") lebih informatif daripada rata-rata.